# CSCI316 — Stage 2A: Spark MLlib (Loan Default Prediction)

**Target:** `default_ind` (1 = default, 0 = no default)

Process One: train **three** Spark MLlib classifiers (Logistic Regression, Random Forest, GBT) with optional CrossValidator tuning, stratified splits, class weights, and validation-set threshold tuning.

**Prerequisites:** run `python stage1.py` first (`output/stage1/data/clean_loan.csv`).

Equivalent script: `python stage2a.py`

**Note:** Only origination-safe features are used; post-loan payment columns are excluded (`LEAKAGE_COLS`).


## Imports and origination-safe feature definitions

This cell loads libraries and defines the **same feature set** as `stage2a.py`:

| Block | Purpose |
|-------|---------|
| `LEAKAGE_COLS` | Post-origination payment/recovery fields — **never** used as predictors |
| `NUMERIC_FEATURES` | Core loan and borrower numerics (e.g. `int_rate`, `term`, `dti`) |
| `ENGINEERED_NUMERIC` | Derived ratios/interactions (e.g. `loan_to_income`, `grade_ord`) |
| `CATEGORICAL_FEATURES` | String columns encoded with StringIndexer + OneHotEncoder |
| `prep_for_modeling_spark()` | Fill categoricals, cast numerics, engineer features, dropna on required cols |

Input: `output/stage1/data/clean_loan.csv` from Stage 1.


In [ ]:
from __future__ import annotations

import os
import sys
import shutil
import socket
import subprocess
import urllib.request
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve

from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when

warnings.filterwarnings("ignore")

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

ROOT = Path.cwd()
DATA_PATH = ROOT / "output" / "stage1" / "data" / "clean_loan.csv"
OUT_DIR = ROOT / "output" / "stage2a"
OUT_DIR.mkdir(parents=True, exist_ok=True)


RANDOM_STATE = 192
LABEL_COL = "default_ind"

LEAKAGE_COLS = {
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_amnt", "out_prncp", "out_prncp_inv", "last_pymnt_d",
    "next_pymnt_d", "last_credit_pull_d",
}

GRADE_TO_ORD = {
    "A": 1.0, "B": 2.0, "C": 3.0, "D": 4.0, "E": 5.0, "F": 6.0, "G": 7.0,
}

NUMERIC_FEATURES = [
    "int_rate", "term", "dti", "annual_inc",
    "delinq_2yrs", "inq_last_6mths", "revol_util",
    "loan_amnt", "open_acc", "pub_rec", "revol_bal",
    "installment", "total_acc",
    "collections_12_mths_ex_med", "acc_now_delinq",
    "tot_coll_amt", "tot_cur_bal", "total_rev_hi_lim",
]

ENGINEERED_NUMERIC = [
    "int_rate_x_term", "loan_to_income", "dti_x_revol_util",
    "grade_ord", "payment_to_income", "revol_bal_ratio", "loan_to_rev_limit",
    "int_rate_x_grade_ord", "dti_x_loan_amnt",
]

CATEGORICAL_FEATURES = [
    "grade", "sub_grade", "home_ownership", "purpose", "verification_status",
    "emp_length", "application_type", "addr_state",
]

# One-hot without grade (grade_ord already encodes risk); used by Stage 2B
CATEGORICAL_FEATURES_OHE = [c for c in CATEGORICAL_FEATURES if c != "grade"]

ALL_NUMERIC = NUMERIC_FEATURES + ENGINEERED_NUMERIC
FEATURE_COLS = ALL_NUMERIC + CATEGORICAL_FEATURES
RAW_LOAD_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [LABEL_COL]

RECOMMENDED_CORE = [
    "int_rate", "term", "dti", "annual_inc",
    "delinq_2yrs", "inq_last_6mths", "revol_util",
]

MOST_RELEVANT_ATTRIBUTES = [
    ("int_rate", "Higher rates reflect riskier loans; strongest numeric linear signal in correlation analysis."),
    ("grade", "LC risk grade; default rate increases from A toward G (default_rate_by_grade.csv)."),
    ("dti", "Debt-to-income measures repayment capacity; standard underwriting variable."),
    ("delinq_2yrs", "Past delinquency is a direct behavioural credit-risk signal."),
    ("inq_last_6mths", "Recent inquiries proxy financial stress and credit seeking."),
    ("revol_util", "High revolving utilisation indicates limited credit headroom."),
    ("loan_amnt", "Loan size relates to exposure; useful with other variables."),
]

LEAST_RELEVANT_ATTRIBUTES = [
    ("id", "Unique loan identifier; no predictive signal."),
    ("member_id", "Unique borrower identifier; must not be used as a model feature."),
    ("emp_title", "Free-text job title; high cardinality and inconsistent labels."),
    ("zip_code", "Redacted/coarse location; weak and unstable risk signal."),
    ("title", "Borrower-entered loan title; noisy unstructured text."),
    ("policy_code", "Administrative flag; near-constant, not borrower risk."),
    ("initial_list_status", "Listing workflow flag; not a credit characteristic."),
]


def _grade_ordinal_pandas(series):
    import pandas as pd

    mapped = series.astype(str).str.strip().str.upper().map(GRADE_TO_ORD)
    return mapped.fillna(4.0)


def _grade_ordinal_spark(df):
    from pyspark.sql.functions import col, lit, when

    expr = when(col("grade") == "A", lit(1.0))
    for letter, val in list(GRADE_TO_ORD.items())[1:]:
        expr = expr.when(col("grade") == letter, lit(val))
    return df.withColumn("grade_ord", expr.otherwise(lit(4.0)))


def engineer_pandas(df):
    import numpy as np

    df = df.copy()
    df["int_rate_x_term"] = df["int_rate"] * df["term"]
    annual = df["annual_inc"].replace(0, np.nan)
    df["loan_to_income"] = df["loan_amnt"] / annual
    df["dti_x_revol_util"] = df["dti"] * df["revol_util"]
    df["grade_ord"] = _grade_ordinal_pandas(df["grade"])
    df["payment_to_income"] = df["installment"] / annual
    rev_lim = df["total_rev_hi_lim"].replace(0, np.nan)
    df["revol_bal_ratio"] = df["revol_bal"] / rev_lim
    df["loan_to_rev_limit"] = df["loan_amnt"] / rev_lim
    df["int_rate_x_grade_ord"] = df["int_rate"] * df["grade_ord"]
    df["dti_x_loan_amnt"] = df["dti"] * df["loan_amnt"]
    # Clip extreme ratios (outliers hurt neural nets)
    df["loan_to_income"] = df["loan_to_income"].clip(upper=5.0)
    df["payment_to_income"] = df["payment_to_income"].clip(upper=0.5)
    df["revol_bal_ratio"] = df["revol_bal_ratio"].clip(upper=1.5)
    return df


def engineer_spark(df):
    from pyspark.sql.functions import col, when

    df = (
        df.withColumn("int_rate_x_term", col("int_rate") * col("term"))
        .withColumn(
            "loan_to_income",
            when(col("annual_inc") > 0, col("loan_amnt") / col("annual_inc")),
        )
        .withColumn("dti_x_revol_util", col("dti") * col("revol_util"))
        .withColumn(
            "payment_to_income",
            when(col("annual_inc") > 0, col("installment") / col("annual_inc")),
        )
        .withColumn(
            "revol_bal_ratio",
            when(col("total_rev_hi_lim") > 0, col("revol_bal") / col("total_rev_hi_lim")),
        )
        .withColumn(
            "loan_to_rev_limit",
            when(col("total_rev_hi_lim") > 0, col("loan_amnt") / col("total_rev_hi_lim")),
        )
    )
    df = _grade_ordinal_spark(df)
    return (
        df.withColumn("int_rate_x_grade_ord", col("int_rate") * col("grade_ord"))
        .withColumn("dti_x_loan_amnt", col("dti") * col("loan_amnt"))
    )


def prep_for_modeling_pandas(df):
    """Load-safe prep: fill categoricals, dropna only on numerics + label."""
    import pandas as pd

    df = df.copy()
    for c in NUMERIC_FEATURES + [LABEL_COL]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in CATEGORICAL_FEATURES:
        df[c] = df[c].fillna("UNKNOWN").astype(str)
    df = engineer_pandas(df)
    required = NUMERIC_FEATURES + ENGINEERED_NUMERIC + [LABEL_COL]
    return df.dropna(subset=required)


def prep_for_modeling_spark(df):
    """Load-safe prep for Spark ML (retains rows when categoricals are missing)."""
    from pyspark.sql.functions import col, lit, when

    for c in CATEGORICAL_FEATURES:
        df = df.withColumn(
            c,
            when(col(c).isNull() | (col(c) == ""), lit("UNKNOWN")).otherwise(col(c)),
        )
    for c in NUMERIC_FEATURES + [LABEL_COL]:
        df = df.withColumn(c, col(c).try_cast("double"))
    df = engineer_spark(df)
    required = NUMERIC_FEATURES + ENGINEERED_NUMERIC + [LABEL_COL]
    return df.dropna(subset=required)


## Configuration, helpers, and evaluation utilities

Optional environment variables (same as `stage2a.py`):

| Variable | Default | Purpose |
|----------|---------|---------|
| `STAGE2A_SAMPLE` | `1.0` | Use `0.1` for a quick dev smoke test only |
| `STAGE2A_TUNE` | `1` | Set `0` to skip CrossValidator (faster) |
| `STAGE2A_UNDERSAMPLE` | off | e.g. `3` = max 3 non-default rows per default in train |
| `STAGE2A_CV_METRIC` | `areaUnderPR` | CV metric: `areaUnderPR` or `areaUnderROC` |
| `STAGE2A_THRESHOLD_STEP` | `0.01` | Validation threshold grid step |

Helper functions defined here include stratified splitting, class weights, threshold tuning (max F1 on **default** class), and metric export helpers.


In [ ]:
SAMPLE_FRAC = float(os.environ.get("STAGE2A_SAMPLE", "1.0"))
ENABLE_TUNE = os.environ.get("STAGE2A_TUNE", "1") != "0"
UNDERSAMPLE_RATIO = os.environ.get("STAGE2A_UNDERSAMPLE")
CV_METRIC = os.environ.get("STAGE2A_CV_METRIC", "areaUnderPR")
THRESHOLD_STEP = float(os.environ.get("STAGE2A_THRESHOLD_STEP", "0.01"))
if CV_METRIC not in ("areaUnderPR", "areaUnderROC"):
    raise ValueError("STAGE2A_CV_METRIC must be areaUnderPR or areaUnderROC")
if not 0.0 < SAMPLE_FRAC <= 1.0:
    raise ValueError("STAGE2A_SAMPLE must be in (0, 1]")
if not 0.0 < THRESHOLD_STEP <= 1.0:
    raise ValueError("STAGE2A_THRESHOLD_STEP must be in (0, 1]")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {DATA_PATH.resolve()}. Run stage1.py first."
    )

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "figure.facecolor": "white",
})

def section(title: str) -> None:
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

def print_table(df: pd.DataFrame, *, pct_cols: tuple = ()) -> None:
    view = df.copy()
    for c in pct_cols:
        if c in view.columns:
            view[c] = (pd.to_numeric(view[c], errors="coerce") * 100).map(
                lambda x: f"{x:.2f}%"
            )
    cols = list(view.columns)
    widths = [max(len(str(c)), *(len(str(v)) for v in view[c].values)) for c in cols]
    fmt = "  ".join(f"{{:{w}}}" for w in widths)
    print(fmt.format(*cols))
    print(fmt.format(*["-" * w for w in widths]))
    for _, row in view.iterrows():
        print(fmt.format(*[str(row[c]) for c in cols]))

def setup_hadoop_winutils(root: Path) -> None:
    if os.name != "nt":
        return
    hadoop_home = root / ".hadoop"
    bin_dir = hadoop_home / "bin"
    bin_dir.mkdir(parents=True, exist_ok=True)
    base = "https://github.com/kontext-tech/winutils/raw/master/hadoop-3.3.0/bin"
    for name in ("winutils.exe", "hadoop.dll"):
        dest = bin_dir / name
        if not dest.is_file():
            print(f"Downloading {name} (optional Hadoop helper)...")
            urllib.request.urlretrieve(f"{base}/{name}", dest)
    os.environ["HADOOP_HOME"] = str(hadoop_home.resolve())
    os.environ["hadoop.home.dir"] = os.environ["HADOOP_HOME"]

def _find_windows_jdk() -> Path | None:
    adoptium = Path(r"C:\Program Files\Eclipse Adoptium")
    if adoptium.is_dir():
        for jdk in sorted(adoptium.glob("jdk-*"), reverse=True):
            if (jdk / "bin" / "java.exe").is_file():
                return jdk
    java_dir = Path(r"C:\Program Files\Java")
    if java_dir.is_dir():
        for jdk in sorted(java_dir.glob("jdk*"), reverse=True):
            if (jdk / "bin" / "java.exe").is_file():
                return jdk
    return None

def build_feature_stages() -> list:
    indexers = [
        StringIndexer(
            inputCol=c,
            outputCol=f"{c}_idx",
            handleInvalid="keep",
        )
        for c in CATEGORICAL_FEATURES
    ]
    idx_cols = [f"{c}_idx" for c in CATEGORICAL_FEATURES]
    ohe_cols = [f"{c}_ohe" for c in CATEGORICAL_FEATURES]
    encoder = OneHotEncoder(
        inputCols=idx_cols,
        outputCols=ohe_cols,
        dropLast=False,
    )
    assembler = VectorAssembler(
        inputCols=ALL_NUMERIC + ohe_cols,
        outputCol="features",
        handleInvalid="keep",
    )
    return indexers + [encoder, assembler]

def add_class_weights(df):
    counts = {
        int(r[LABEL_COL]): int(r["count"])
        for r in df.groupBy(LABEL_COL).count().collect()
    }
    n0 = counts.get(0, 0)
    n1 = counts.get(1, 0)
    if n1 == 0:
        raise ValueError("No positive (default) labels in dataset.")
    ratio = n0 / n1
    weighted = df.withColumn(
        "weight",
        when(col(LABEL_COL) == 1.0, lit(ratio)).otherwise(lit(1.0)),
    )
    default_rate = n1 / (n0 + n1)
    return weighted, default_rate, n0, n1

def stratified_train_val_test(df, fracs: tuple[float, float, float], seed: int):
    """Stratified split: preserve default rate in train / val / test."""
    train_frac, val_frac, test_frac = fracs
    if abs(train_frac + val_frac + test_frac - 1.0) > 1e-6:
        raise ValueError("Split fractions must sum to 1.0")
    train_parts, val_parts, test_parts = [], [], []
    for label in (0.0, 1.0):
        subset = df.filter(col(LABEL_COL) == label)
        tr, va, te = subset.randomSplit([train_frac, val_frac, test_frac], seed=seed)
        train_parts.append(tr)
        val_parts.append(va)
        test_parts.append(te)
    train_out = train_parts[0]
    val_out = val_parts[0]
    test_out = test_parts[0]
    for part in train_parts[1:]:
        train_out = train_out.unionByName(part)
    for part in val_parts[1:]:
        val_out = val_out.unionByName(part)
    for part in test_parts[1:]:
        test_out = test_out.unionByName(part)
    return train_out, val_out, test_out

def print_split_balance(name: str, df, n: int | None = None) -> None:
    if n is None:
        n = df.count()
    n1 = sum(
        int(r["count"])
        for r in df.groupBy(LABEL_COL).count().collect()
        if float(r[LABEL_COL]) == 1.0
    )
    rate = n1 / n if n else 0.0
    print(f"  {name}: {n:,} rows  |  default rate: {rate:.2%}")

def undersample_train(df, max_ratio: float):
    """Keep all defaults; subsample non-defaults to max_ratio * n_default."""
    defaults = df.filter(col(LABEL_COL) == 1.0)
    non_defaults = df.filter(col(LABEL_COL) == 0.0)
    n1 = defaults.count()
    n0 = non_defaults.count()
    target_n0 = int(min(n0, max_ratio * n1))
    if target_n0 >= n0:
        return df
    frac = target_n0 / n0
    sampled_0 = non_defaults.sample(withReplacement=False, fraction=frac, seed=RANDOM_STATE)
    out = sampled_0.unionByName(defaults)
    print(f"Undersampled train: {n0:,} -> {target_n0:,} non-default rows (ratio {max_ratio}:1)")
    return out

def confusion_counts(predictions, pred_col: str = "prediction") -> dict[str, int]:
    rows = (
        predictions.select(LABEL_COL, pred_col)
        .groupBy(LABEL_COL, pred_col)
        .count()
        .collect()
    )
    counts = {
        (int(r[LABEL_COL]), float(r[pred_col])): int(r["count"])
        for r in rows
    }
    tp = counts.get((1, 1.0), 0)
    tn = counts.get((0, 0.0), 0)
    fp = counts.get((0, 1.0), 0)
    fn = counts.get((1, 0.0), 0)
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn}

def binary_default_metrics(cm: dict[str, int]) -> dict[str, float]:
    tp, fp, fn = cm["tp"], cm["fp"], cm["fn"]
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    return {
        "precision_default": prec,
        "recall_default": rec,
        "f1_default": f1,
    }

def tune_threshold(val_predictions, thresholds: np.ndarray | None = None) -> tuple[float, float]:
    """Pick threshold on validation set maximizing F1 for default class."""
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, THRESHOLD_STEP)
    pdf = val_predictions.select(LABEL_COL, "probability").toPandas()
    pdf["prob_default"] = pdf["probability"].apply(lambda v: float(v[1]))
    y = pdf[LABEL_COL].astype(int).values
    scores = pdf["prob_default"].values
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tp = ((pred == 1) & (y == 1)).sum()
        fp = ((pred == 1) & (y == 0)).sum()
        fn = ((pred == 0) & (y == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1

def evaluate_predictions(
    name: str,
    predictions,
    pred_col: str,
    train_n: int,
    test_n: int,
    threshold: float | None = None,
) -> dict:
    binary_eval = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )
    auc = binary_eval.evaluate(predictions)
    auc_pr = binary_eval.setMetricName("areaUnderPR").evaluate(predictions)
    cm = confusion_counts(predictions, pred_col)
    multi = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL,
        predictionCol=pred_col,
    )
    accuracy = multi.setMetricName("accuracy").evaluate(predictions)
    precision_w = multi.setMetricName("weightedPrecision").evaluate(predictions)
    recall_w = multi.setMetricName("weightedRecall").evaluate(predictions)
    bin_m = binary_default_metrics(cm)
    row = {
        "model": name,
        "pred_col": pred_col,
        "threshold": threshold if threshold is not None else 0.5,
        "accuracy": accuracy,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,
        "auc_roc": auc,
        "auc_pr": auc_pr,
        "train_n": train_n,
        "test_n": test_n,
        **cm,
        **bin_m,
    }
    return row

def roc_points(y_true: np.ndarray, y_score: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(-y_score)
    y = y_true[order].astype(float)
    P = y.sum()
    N = len(y) - P
    if P == 0 or N == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])
    tps = np.cumsum(y)
    fps = np.cumsum(1.0 - y)
    tpr = np.concatenate([[0.0], tps / P])
    fpr = np.concatenate([[0.0], fps / N])
    return fpr, tpr

def collect_roc(name: str, predictions, roc_data: list) -> None:
    pdf = predictions.select(LABEL_COL, "probability").toPandas()
    y_true = pdf[LABEL_COL].astype(int).values
    y_score = pdf["probability"].apply(lambda v: float(v[1])).values
    fpr, tpr = roc_points(y_true, y_score)
    auc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    ).evaluate(predictions)
    roc_data.append((name, fpr, tpr, auc))

def collect_pr(name: str, predictions, pr_data: list) -> None:
    pdf = predictions.select(LABEL_COL, "probability").toPandas()
    y_true = pdf[LABEL_COL].astype(int).values
    y_score = pdf["probability"].apply(lambda v: float(v[1])).values
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    pr_data.append((name, precision, recall, ap))

def plot_roc_curves(roc_data: list) -> None:
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = ["#3498db", "#27ae60", "#9b59b6", "#e67e22", "#1abc9c"]
    for (name, fpr, tpr, auc), color in zip(roc_data, colors):
        ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, linewidth=2)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curves — test set (Spark MLlib)")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    path = OUT_DIR / "roc_curves.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

def plot_pr_curves(pr_data: list) -> None:
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = ["#3498db", "#27ae60", "#9b59b6", "#e67e22", "#1abc9c"]
    for (name, precision, recall, ap), color in zip(pr_data, colors):
        ax.plot(recall, precision, label=f"{name} (AP={ap:.3f})", color=color, linewidth=2)
    ax.set_xlabel("Recall (default class)")
    ax.set_ylabel("Precision (default class)")
    ax.set_title("Precision–recall curves — test set (Spark MLlib)")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    path = OUT_DIR / "pr_curves.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

def plot_confusion_heatmap(
    cm: dict[str, int],
    *,
    model_name: str,
    threshold: float,
    path: Path,
) -> None:
    matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]], dtype=float)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Pred: no default", "Pred: default"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Actual: no default", "Actual: default"])
    for i in range(2):
        for j in range(2):
            val = int(matrix[i, j])
            color = "white" if matrix[i, j] > matrix.max() * 0.5 else "black"
            ax.text(j, i, f"{val:,}", ha="center", va="center", color=color, fontsize=11)
    ax.set_title(f"Confusion matrix — {model_name}\n(tuned threshold = {threshold:.3f})")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

def feature_names_from_preprocess(preprocess_model) -> list[str]:
    names: list[str] = []
    indexers = {
        s.getOutputCol(): s
        for s in preprocess_model.stages
        if hasattr(s, "labels")
    }
    assembler = next(
        s for s in preprocess_model.stages if "VectorAssembler" in type(s).__name__
    )
    for col_name in assembler.getInputCols():
        if col_name in ALL_NUMERIC:
            names.append(col_name)
        elif col_name.endswith("_ohe"):
            cat = col_name.replace("_ohe", "")
            indexer = indexers.get(f"{cat}_idx")
            if indexer is not None:
                for lab in indexer.labels:
                    names.append(f"{cat}={lab}")
            else:
                names.append(col_name)
    return names

def export_tree_feature_importance(
    tree_model,
    preprocess_model,
    model_label: str,
    *,
    top_n: int = 20,
) -> None:
    imp = tree_model.featureImportances.toArray()
    names = feature_names_from_preprocess(preprocess_model)
    if len(names) != len(imp):
        names = [f"f{i}" for i in range(len(imp))]
    fi_df = (
        pd.DataFrame({"feature": names, "importance": imp})
        .sort_values("importance", ascending=False)
        .head(top_n)
    )
    slug = model_label.lower()
    csv_path = OUT_DIR / f"feature_importance_{slug}.csv"
    fi_df.to_csv(csv_path, index=False)
    print("Saved:", csv_path)

    fig, ax = plt.subplots(figsize=(8, max(4.0, 0.28 * len(fi_df))))
    plot_df = fi_df.iloc[::-1]
    ax.barh(plot_df["feature"], plot_df["importance"], color="#27ae60")
    ax.set_xlabel("Importance (Gini)")
    ax.set_title(f"Top {len(fi_df)} features — {model_label}")
    plt.tight_layout()
    png_path = OUT_DIR / f"feature_importance_{slug}.png"
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", png_path)

def fit_with_optional_cv(estimator, train_df, param_grid, label: str):
    if not ENABLE_TUNE or not param_grid:
        print(f"  Fitting {label} (fixed hyperparameters)...")
        return estimator.fit(train_df)
    folds = 5 if SAMPLE_FRAC >= 0.5 else 3
    print(f"  CrossValidator {label} ({folds} folds, metric={CV_METRIC})...")
    evaluator = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName=CV_METRIC,
    )
    cv = CrossValidator(
        estimator=estimator,
        estimatorParamMaps=param_grid,
        evaluator=evaluator,
        numFolds=folds,
        seed=RANDOM_STATE,
        parallelism=2,
    )
    cv_model = cv.fit(train_df)
    return cv_model.bestModel

# --- Spark setup ---


## 2.1 Spark setup

Start a local Spark session for MLlib. On Windows, Java 11/17 is required; host is bound to `127.0.0.1` if the PC name contains `_`.

Outputs from this stage are written under `output/stage2a/`.


In [ ]:
# Ensure Spark workers use this interpreter (avoids Windows Store python alias)
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

if not shutil.which("java"):
    jdk_home = _find_windows_jdk()
    if jdk_home is None:
        raise RuntimeError(
            "Java not found. Install Temurin JDK 17, restart terminal, re-run. "
            "https://adoptium.net/"
        )
    os.environ["JAVA_HOME"] = str(jdk_home)
    os.environ["PATH"] = str(jdk_home / "bin") + os.pathsep + os.environ.get("PATH", "")

if "_" in socket.gethostname():
    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

setup_hadoop_winutils(ROOT)
print("Java:", subprocess.check_output(
    ["java", "-version"], stderr=subprocess.STDOUT, text=True
).splitlines()[0])

SPARK_TMP = ROOT / ".spark-tmp"
SPARK_TMP.mkdir(exist_ok=True)

spark = (
    SparkSession.builder
    .appName("CSCI316_Stage2A_MLlib")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.local.dir", str(SPARK_TMP))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

def add_prob_default(predictions):
    return predictions.withColumn(
        "prob_default",
        vector_to_array(col("probability"))[1],
    )

def with_prob_default(predictions):
    return add_prob_default(predictions)

def apply_threshold_tuned(predictions, threshold: float, out_col: str = "prediction_tuned"):
    pred = add_prob_default(predictions)
    return pred.withColumn(
        out_col,
        when(col("prob_default") >= lit(threshold), lit(1.0)).otherwise(lit(0.0)),
    )

print("CSCI316 Stage 2A — Spark MLlib (enhanced)")
print(f"Run started: {datetime.now():%Y-%m-%d %H:%M}")
print(f"Input: {DATA_PATH.relative_to(ROOT)}  |  Output: {OUT_DIR.relative_to(ROOT)}/")
print(f"Tuning: {'on' if ENABLE_TUNE else 'off (STAGE2A_TUNE=0)'}")
print(f"CV metric: {CV_METRIC}  |  threshold step: {THRESHOLD_STEP}")
if SAMPLE_FRAC < 1.0:
    print(f"STAGE2A_SAMPLE={SAMPLE_FRAC} (dev mode)")
if UNDERSAMPLE_RATIO:
    print(f"STAGE2A_UNDERSAMPLE={UNDERSAMPLE_RATIO}")
print(f"Spark {spark.version}")

# --- Load data ---


## 2.2 Load data and engineer features

Load the cleaned Stage 1 CSV, keep only origination-safe columns, and run `prep_for_modeling_spark()`:

- Cast numerics and fill missing categoricals with `UNKNOWN`
- Build engineered features (`int_rate_x_term`, `grade_ord`, etc.)
- Drop rows with missing values in required numeric/engineered columns

**No leakage:** payment history columns from `LEAKAGE_COLS` are not in `RAW_LOAD_COLS`.


## Stage 1 feature relevance (7 most / 7 least)

Tables produced in Stage 1; modelling uses the expanded origination-safe feature set defined in `stage2a.py`.


In [ ]:
STAGE1_DIR = ROOT / "output" / "stage1"
most_rel = pd.read_csv(STAGE1_DIR / "tables" / "feature_relevance_most_relevant.csv")
least_rel = pd.read_csv(STAGE1_DIR / "tables" / "feature_relevance_least_relevant.csv")
print("7 most relevant attributes:")
display(most_rel)
print("7 least relevant attributes:")
display(least_rel)


In [ ]:
raw_df = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_PATH))
missing = [c for c in RAW_LOAD_COLS if c not in raw_df.columns]
if missing:
    raise ValueError(f"Missing columns in clean_loan.csv: {missing}")

model_df = prep_for_modeling_spark(raw_df.select(*RAW_LOAD_COLS))
if SAMPLE_FRAC < 1.0:
    model_df = model_df.sample(withReplacement=False, fraction=SAMPLE_FRAC, seed=RANDOM_STATE)
model_df = model_df.persist(StorageLevel.MEMORY_AND_DISK)
n_clean = model_df.count()
print(f"Rows after dropna{' + sample' if SAMPLE_FRAC < 1.0 else ''}: {n_clean:,}")
print(f"Features ({len(FEATURE_COLS)}): {', '.join(FEATURE_COLS)}")

model_df, default_rate, n0, n1 = add_class_weights(model_df)
print(f"Class balance — no default: {n0:,}  |  default: {n1:,}  |  rate: {default_rate:.2%}")

# --- Splits: stratified 64% train / 16% val / 20% test ---


## 2.3 Train / validation / test split (stratified)

Split **64% / 16% / 20%** with **stratification** by `default_ind` so each fold keeps a similar default rate (~4–5%).

- **Train** — fit models and feature pipeline
- **Validation** — CrossValidator and **threshold tuning** (max F1 for default class)
- **Test** — held out until final evaluation

**Class weights** up-weight the minority default class during training.


In [ ]:
train_df, val_df, test_df = stratified_train_val_test(
    model_df, (0.64, 0.16, 0.20), RANDOM_STATE
)
if UNDERSAMPLE_RATIO:
    train_df = undersample_train(train_df, float(UNDERSAMPLE_RATIO))
    train_df, _, _, _ = add_class_weights(train_df)

train_df = train_df.cache()
val_df = val_df.cache()
test_df = test_df.cache()
train_n = train_df.count()
val_n = val_df.count()
test_n = test_df.count()
print(f"Train: {train_n:,}  |  Val: {val_n:,}  |  Test: {test_n:,}")
print_split_balance("train", train_df, train_n)
print_split_balance("val", val_df, val_n)
print_split_balance("test", test_df, test_n)

feature_stages = build_feature_stages()
metrics_rows: list[dict] = []
tuned_rows: list[dict] = []
roc_data: list = []
pr_data: list = []
best_predictions = None
best_name = ""

# --- Logistic Regression ---
lr_est = LogisticRegression(
    featuresCol="scaled_features",
    labelCol=LABEL_COL,
    weightCol="weight",
    maxIter=100,
    regParam=0.01,
)
lr_pipeline = Pipeline(stages=feature_stages + [
    StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=True),
    lr_est,
])
lr_grid_builder = (
    ParamGridBuilder()
    .addGrid(lr_est.regParam, [0.001, 0.01, 0.1])
    .addGrid(lr_est.elasticNetParam, [0.0, 0.5])
)
if SAMPLE_FRAC >= 0.5:
    lr_grid = lr_grid_builder.addGrid(lr_est.maxIter, [100, 200]).build()
else:
    lr_grid = lr_grid_builder.build()


## Model 1 — Logistic Regression

Baseline linear classifier on **scaled** features (`StandardScaler`).

- Optional **CrossValidator** over `regParam`, `elasticNetParam`, `maxIter`
- Default threshold 0.5 plus **tuned** threshold from validation F1
- Compare ROC-AUC / PR-AUC and default-class precision, recall, F1


In [ ]:
lr_model = fit_with_optional_cv(lr_pipeline, train_df, lr_grid, "LogisticRegression")
lr_val = lr_model.transform(val_df)
lr_test = lr_model.transform(test_df)
best_t, val_f1 = tune_threshold(lr_val)
print(f"  Validation: best threshold={best_t:.3f}  F1(default)={val_f1:.4f}")
lr_tuned = apply_threshold_tuned(lr_test, best_t)
m_lr = evaluate_predictions("LogisticRegression", lr_test, "prediction", train_n, test_n)
m_lr_t = evaluate_predictions("LogisticRegression", lr_tuned, "prediction_tuned", train_n, test_n, best_t)
metrics_rows.append(m_lr)
tuned_rows.append({**m_lr_t, "model": "LogisticRegression", "best_threshold": best_t, "val_f1_default": val_f1})
collect_roc("LogisticRegression", lr_test, roc_data)
collect_pr("LogisticRegression", lr_test, pr_data)
print(
    f"  LR test AUC={m_lr['auc_roc']:.4f}  PR-AUC={m_lr['auc_pr']:.4f}  "
    f"tuned recall_default={m_lr_t['recall_default']:.4f}"
)

# --- Random Forest ---


## Model 2 — Random Forest

Ensemble of decision trees on **unscaled** encoded features (trees are scale-invariant).

- Optional CV over `numTrees` and `maxDepth`
- Feature importance exported for interpretation slides
- Threshold tuned on validation set for the default class


In [ ]:
preprocess = Pipeline(stages=feature_stages)
preprocess_model = preprocess.fit(train_df)
train_feat = preprocess_model.transform(train_df).cache()
val_feat = preprocess_model.transform(val_df).cache()
test_feat = preprocess_model.transform(test_df).cache()

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    weightCol="weight",
    numTrees=100,
    maxDepth=10,
    seed=RANDOM_STATE,
)
if SAMPLE_FRAC >= 0.5:
    rf_grid = (
        ParamGridBuilder()
        .addGrid(rf.numTrees, [100, 150])
        .addGrid(rf.maxDepth, [8, 10, 12])
        .build()
    )
else:
    rf_grid = (
        ParamGridBuilder()
        .addGrid(rf.numTrees, [80, 100])
        .addGrid(rf.maxDepth, [8, 10])
        .build()
    )
rf_model = (
    fit_with_optional_cv(rf, train_feat, rf_grid, "RandomForest")
    if ENABLE_TUNE else rf.fit(train_feat)
)
rf_val = rf_model.transform(val_feat)
rf_test = rf_model.transform(test_feat)
best_t, val_f1 = tune_threshold(rf_val)
print(f"  Validation: best threshold={best_t:.3f}  F1(default)={val_f1:.4f}")
rf_tuned = apply_threshold_tuned(rf_test, best_t)
m_rf = evaluate_predictions("RandomForest", rf_test, "prediction", train_n, test_n)
m_rf_t = evaluate_predictions("RandomForest", rf_tuned, "prediction_tuned", train_n, test_n, best_t)
metrics_rows.append(m_rf)
tuned_rows.append({**m_rf_t, "model": "RandomForest", "best_threshold": best_t, "val_f1_default": val_f1})
collect_roc("RandomForest", rf_test, roc_data)
collect_pr("RandomForest", rf_test, pr_data)
print(
    f"  RF test AUC={m_rf['auc_roc']:.4f}  PR-AUC={m_rf['auc_pr']:.4f}  "
    f"tuned recall_default={m_rf_t['recall_default']:.4f}"
)

# --- GBT ---


## Model 3 — Gradient Boosted Trees

Sequential tree boosting; often strong on tabular credit data.

- Optional CV over `maxDepth`, `maxIter`, `stepSize`, `subsamplingRate`
- Same evaluation and threshold tuning as RF
- Feature importance exported for slides


In [ ]:
gbt = GBTClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    weightCol="weight",
    maxIter=50,
    maxDepth=6,
    stepSize=0.1,
    subsamplingRate=1.0,
    seed=RANDOM_STATE,
)
if SAMPLE_FRAC >= 0.5:
    gbt_grid = (
        ParamGridBuilder()
        .addGrid(gbt.maxDepth, [5, 6, 7, 8])
        .addGrid(gbt.maxIter, [50, 80, 100])
        .addGrid(gbt.stepSize, [0.05, 0.1])
        .addGrid(gbt.subsamplingRate, [0.8, 1.0])
        .build()
    )
else:
    gbt_grid = (
        ParamGridBuilder()
        .addGrid(gbt.maxDepth, [5, 7])
        .addGrid(gbt.maxIter, [50, 80])
        .addGrid(gbt.stepSize, [0.1])
        .build()
    )
gbt_model = (
    fit_with_optional_cv(gbt, train_feat, gbt_grid, "GBT")
    if ENABLE_TUNE else gbt.fit(train_feat)
)
gbt_val = gbt_model.transform(val_feat)
gbt_test = gbt_model.transform(test_feat)
best_t, val_f1 = tune_threshold(gbt_val)
print(f"  Validation: best threshold={best_t:.3f}  F1(default)={val_f1:.4f}")
gbt_tuned = apply_threshold_tuned(gbt_test, best_t)
m_gbt = evaluate_predictions("GBT", gbt_test, "prediction", train_n, test_n)
m_gbt_t = evaluate_predictions("GBT", gbt_tuned, "prediction_tuned", train_n, test_n, best_t)
metrics_rows.append(m_gbt)
tuned_rows.append({**m_gbt_t, "model": "GBT", "best_threshold": best_t, "val_f1_default": val_f1})
collect_roc("GBT", gbt_test, roc_data)
collect_pr("GBT", gbt_test, pr_data)
print(
    f"  GBT test AUC={m_gbt['auc_roc']:.4f}  PR-AUC={m_gbt['auc_pr']:.4f}  "
    f"tuned recall_default={m_gbt_t['recall_default']:.4f}"
)

# Best by tuned F1 default
tuned_df = pd.DataFrame(tuned_rows)
best_tuned_row = tuned_df.loc[tuned_df["f1_default"].idxmax()]
best_name = best_tuned_row["model"]
if best_name == "LogisticRegression":
    best_predictions = lr_tuned
elif best_name == "RandomForest":
    best_predictions = rf_tuned
else:
    best_predictions = gbt_tuned

# --- Export ---


## 2.8 Export results

Save artefacts under `output/stage2a/`:

| File | Description |
|------|-------------|
| `model_metrics.csv` | Test metrics @ threshold 0.5 |
| `model_metrics_tuned.csv` | Test metrics @ validation-tuned threshold (**primary**) |
| `threshold_tuning.csv` | Best threshold per model |
| `confusion_matrices*.csv` | TP/TN/FP/FN |
| `roc_curves.png` / `pr_curves.png` | Test-set curves (3 models) |
| `feature_importance_*.png` | Top RF/GBT features |
| `stage2a_executive_summary.txt` | Short summary for slides |

**Critical analysis:** prefer PR-AUC and tuned default-class recall/F1 over accuracy alone.


In [ ]:
metrics_df = pd.DataFrame(metrics_rows)
metrics_path = OUT_DIR / "model_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print("Saved:", metrics_path)

tuned_path = OUT_DIR / "model_metrics_tuned.csv"
tuned_df.to_csv(tuned_path, index=False)
print("Saved:", tuned_path)

threshold_path = OUT_DIR / "threshold_tuning.csv"
pd.DataFrame([
    {
        "model": r["model"],
        "best_threshold": r["best_threshold"],
        "val_f1_default": r["val_f1_default"],
        "test_f1_default": r["f1_default"],
    }
    for r in tuned_rows
]).to_csv(threshold_path, index=False)
print("Saved:", threshold_path)

cm_path = OUT_DIR / "confusion_matrices.csv"
pd.DataFrame([
    {"model": r["model"], "tp": r["tp"], "tn": r["tn"], "fp": r["fp"], "fn": r["fn"]}
    for r in metrics_rows
]).to_csv(cm_path, index=False)
print("Saved:", cm_path)

cm_tuned_path = OUT_DIR / "confusion_matrices_tuned.csv"
pd.DataFrame([
    {
        "model": r["model"],
        "threshold": r["best_threshold"],
        "tp": r["tp"], "tn": r["tn"], "fp": r["fp"], "fn": r["fn"],
    }
    for r in tuned_rows
]).to_csv(cm_tuned_path, index=False)
print("Saved:", cm_tuned_path)

plot_roc_curves(roc_data)
plot_pr_curves(pr_data)

best_cm = {
    "tp": int(best_tuned_row["tp"]),
    "tn": int(best_tuned_row["tn"]),
    "fp": int(best_tuned_row["fp"]),
    "fn": int(best_tuned_row["fn"]),
}
plot_confusion_heatmap(
    best_cm,
    model_name=str(best_name),
    threshold=float(best_tuned_row["best_threshold"]),
    path=OUT_DIR / "confusion_matrix_best_tuned.png",
)

export_tree_feature_importance(rf_model, preprocess_model, "RandomForest")
export_tree_feature_importance(gbt_model, preprocess_model, "GBT")

print("\nModel comparison @ default threshold 0.5 (test):")
print_table(
    metrics_df[
        [
            "model", "auc_roc", "auc_pr", "accuracy",
            "precision_default", "recall_default", "f1_default",
        ]
    ].round(4)
)
print("\nModel comparison @ tuned threshold (test):")
print_table(
    tuned_df[
        ["model", "best_threshold", "precision_default", "recall_default", "f1_default"]
    ].round(4)
)

if best_predictions is not None:
    sample_path = OUT_DIR / "predictions_sample.csv"
    (
        with_prob_default(best_predictions)
        .select(LABEL_COL, "prediction_tuned", "prob_default")
        .limit(1000)
        .toPandas()
        .to_csv(sample_path, index=False)
    )
    print("Saved:", sample_path, f"(best tuned: {best_name})")

best_auc_row = metrics_df.loc[metrics_df["auc_roc"].idxmax()]
summary_lines = [
    "CSCI316 Stage 2A — Spark MLlib summary (enhanced)",
    f"Run: {datetime.now():%Y-%m-%d %H:%M}",
    f"Train: {train_n:,}  |  Val: {val_n:,}  |  Test: {test_n:,}",
    f"Default rate: {default_rate:.2%}",
    f"Features ({len(FEATURE_COLS)}): {', '.join(FEATURE_COLS)}",
    f"Engineered: {', '.join(ENGINEERED_NUMERIC)}",
    "No post-origination payment features used.",
    f"Hyperparameter tuning: {'enabled' if ENABLE_TUNE else 'disabled'}",
    f"CV metric: {CV_METRIC}  |  stratified split  |  threshold step: {THRESHOLD_STEP}",
    "",
    "Test @ threshold 0.5:",
]
for _, r in metrics_df.iterrows():
    summary_lines.append(
        f"  {r['model']}: ROC-AUC={r['auc_roc']:.4f}  PR-AUC={r['auc_pr']:.4f}  "
        f"recall_default={r['recall_default']:.4f}  precision_default={r['precision_default']:.4f}"
    )
summary_lines.append("")
summary_lines.append("Test @ tuned threshold (max F1 on validation):")
for _, r in tuned_df.iterrows():
    summary_lines.append(
        f"  {r['model']} (t={r['best_threshold']:.3f}): F1={r['f1_default']:.4f}  "
        f"recall_default={r['recall_default']:.4f}  precision_default={r['precision_default']:.4f}"
    )
summary_lines.extend([
    "",
    f"Best AUC: {best_auc_row['model']} ({best_auc_row['auc_roc']:.4f})",
    f"Best tuned F1 (default class): {best_name} ({best_tuned_row['f1_default']:.4f})",
])
summary_path = OUT_DIR / "stage2a_executive_summary.txt"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print("Saved:", summary_path)


## Stage 2A — copy to slides

Console summary of the best models for your presentation **Results** and **Discussion** slides.

Use **tuned** metrics when reporting how well the model catches defaults.


In [ ]:
print(f"Default rate: {default_rate:.1%}  |  Models: LR, RF, GBT (Spark MLlib)")
print(f"Best AUC: {best_auc_row['model']} ({best_auc_row['auc_roc']:.3f})")
print(
    f"Best tuned for defaults: {best_name}  "
    f"recall={best_tuned_row['recall_default']:.3f}  "
    f"precision={best_tuned_row['precision_default']:.3f}  "
    f"threshold={best_tuned_row['best_threshold']:.3f}"
)
print("No post-origination payment features used as predictors.")

model_df.unpersist()
train_df.unpersist()
val_df.unpersist()
test_df.unpersist()
train_feat.unpersist()
val_feat.unpersist()
test_feat.unpersist()
spark.stop()
print("\nStage 2A complete.")


## Stage 2A findings (for slides)

- Three Spark MLlib models: LR, RF, GBT.
- Report **ROC-AUC / PR-AUC** and **default-class precision, recall, F1** at the **validation-tuned** threshold.
- Accuracy @ 0.5 alone is misleading on imbalanced data.
- No post-origination payment features were used as predictors.
